# 蘇東坡 AI 雙模組協作中轉站（Colab T4 版）

**架構**：Gemini 查史實 → 組裝結構化提示詞 → 本地東坡 LoRA 模型以蘇東坡口吻回答。

這是「RAG 式認知邊界」路線：邊界不靠權重死記，而靠外部 Gemini 即時查證史實、再以提示詞指引東坡模型「界外裝作不知」。

---
### 執行環境
- **平台**：Google Colab，硬體加速器選 **T4 GPU**（執行階段 → 變更執行階段類型）。
- **顯存**：東坡 8B 以 4-bit 量化載入約佔 5–6GB，T4 的 15GB 足夠（Gemini 走 API 不佔顯存）。
- **金鑰**：用 Colab Secrets 存放，**不寫進程式碼**。

### ⚠️ 安全須知（務必先做）
1. 原 `.py` 內那兩把明碼金鑰已視同外洩，**請先到 Google AI Studio 撤銷、另產生新金鑰**。
2. 點左側 🔑（Secrets），新增一筆：名稱 `GEMINI_API_KEY`，值填你的新金鑰，並開啟「Notebook access」。

### 依賴
`torch`、`transformers`、`peft`、`accelerate`、`bitsandbytes`（Colab 內建大多可用）、`google-genai`（新版 SDK，下面會裝）。

---
## [1] 安裝 / 更新套件

In [ ]:
# google-genai 是新版 SDK（取代舊的 google-generativeai）。
# 其餘套件 Colab 多半內建；若 transformers 太舊不認得 qwen3，解開下行更新。
%pip install -q -U google-genai
# %pip install -q -U transformers peft accelerate bitsandbytes
print("套件安裝完成")

---
## [2] 讀取金鑰（Colab Secrets）
先確認左側 🔑 已建立 `GEMINI_API_KEY`。

In [ ]:
from google.colab import userdata

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    assert GEMINI_API_KEY and GEMINI_API_KEY.strip(), "金鑰為空"
    print("✅ 已從 Colab Secrets 讀到 GEMINI_API_KEY（長度：%d）" % len(GEMINI_API_KEY))
except Exception as e:
    raise RuntimeError(
        "讀不到金鑰。請點左側 🔑 新增名為 GEMINI_API_KEY 的 Secret，"
        "填入你的新金鑰並開啟 Notebook access。\n原始錯誤：%s" % e
    )

---
## [3] 掛載 Google Drive 並設定模型路徑
東坡底模與 adapter 都放在 Drive，請依實際位置修改下面兩個路徑。

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

# === 依你 Drive 實際位置修改 ===
BASE_MODEL_PATH = "/content/drive/MyDrive/FJU_dongpoProject/dongPo/models/DongPo-Base"
LORA_WEIGHT_PATH = "/content/drive/MyDrive/FJU_dongpoProject/dongPo/models/DongPo-Adapter"
# ==============================

# 驗證路徑與必要檔案
assert os.path.isdir(BASE_MODEL_PATH), f"找不到底模資料夾：{BASE_MODEL_PATH}"
assert os.path.isdir(LORA_WEIGHT_PATH), f"找不到 adapter 資料夾：{LORA_WEIGHT_PATH}"
assert os.path.isfile(os.path.join(LORA_WEIGHT_PATH, "adapter_config.json")), "adapter 缺 adapter_config.json"
_w = any(os.path.isfile(os.path.join(LORA_WEIGHT_PATH, f)) for f in ("adapter_model.safetensors", "adapter_model.bin"))
assert _w, "adapter 缺 adapter_model.safetensors / .bin"
print("路徑正確讀取完畢")

---
## [4] 匯入與 4-bit 量化設定（T4 自動用 float16）

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from google import genai
from google.genai import types

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("裝置：", DEVICE)
if DEVICE == "cuda":
    name = torch.cuda.get_device_name(0)
    major, _ = torch.cuda.get_device_capability()
    print("GPU：", name)
    # T4 (capability 7.5) 不支援 bfloat16 → 用 float16；3070/A100 等 (>=8) 用 bfloat16
    COMPUTE_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
    print("compute_dtype：", "bfloat16" if COMPUTE_DTYPE == torch.bfloat16 else "float16")
else:
    COMPUTE_DTYPE = torch.float16
    print("⚠️ 沒抓到 GPU。請到『執行階段 → 變更執行階段類型』選 T4 GPU。")

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
print("BNB 量化設定就緒")

---
## [5] 初始化 Gemini（史實查核模組）

In [ ]:
GEMINI_SYSTEM = (
    "你是一個專屬蘇東坡的歷史背景知識庫。請提供客觀、簡明的史實。"
    "【嚴格禁止】自稱人工智慧或AI。"
    "若使用者詢問蘇東坡的第一人稱問題（如：你喜歡吃什麼？），請直接輸出蘇東坡的歷史生平事實"
    "（如：蘇軾在黃州喜歡吃豬肉、竹筍等）。"
    "若使用者詢問後世歷史，請直接給出該歷史事件的客觀簡介。"
)

gclient = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL_NAME = "gemini-2.5-flash"

def get_historical_facts(user_question: str) -> str:
    """用 Gemini 查客觀史實。新版 SDK 寫法，含簡單錯誤處理。"""
    try:
        resp = gclient.models.generate_content(
            model=GEMINI_MODEL_NAME,
            contents=user_question,
            config=types.GenerateContentConfig(system_instruction=GEMINI_SYSTEM),
        )
        return (resp.text or "").strip()
    except Exception as exc:
        msg = str(exc)
        if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
            return "（史實查詢觸發流量限制 429，請稍後再試，或於 Secrets 換一把金鑰）"
        return f"（史實查詢失敗：{msg}）"

# 小測試（會真的呼叫一次 API，確認金鑰可用）
print("Gemini 測試回應：")
print(get_historical_facts("蘇軾在黃州的飲食"))

---
## [6] 載入東坡底模 + 融合 LoRA

In [ ]:
print("載入底模中…（第一次較久）")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=BNB_CONFIG,
    device_map="auto",
    trust_remote_code=True,
)
print("融合 LoRA 中…")
model = PeftModel.from_pretrained(base_model, LORA_WEIGHT_PATH)
model.eval()
print("✅ 東坡 + adapter 就緒")
if torch.cuda.is_available():
    print(f"顯存佔用：{torch.cuda.memory_allocated()/1024**3:.2f} GB")

---
## [7] 確認對話格式（重要）
東坡用的是**非標準** Human/Assistant 格式（不是標準 ChatML）。`apply_chat_template` 會讀東坡 tokenizer 自帶的 template。
這格把實際組出來的提示詞印出來，**你親眼確認格式對不對**，再往下用。

In [ ]:
_demo_msgs = [
    {"role": "system", "content": "（測試 system）"},
    {"role": "user", "content": "（測試 user 問題）"},
]
_demo = tokenizer.apply_chat_template(_demo_msgs, tokenize=False, add_generation_prompt=True)
print("=== apply_chat_template 實際輸出 ===")
print(_demo)
print("=== 檢查 ===")
print("含 <|im_start|>（標準ChatML特徵）：", "<|im_start|>" in _demo)
print("含 Human:/Assistant:（東坡自訂格式特徵）：", ("Human:" in _demo or "Assistant:" in _demo))
print("\n若是 Human/Assistant 格式即與底模一致，可放心使用。")

---
## [8] 組裝提示詞 + 本地推論

In [ ]:
SUDONGPO_SYSTEM = (
    "你現在是宋代文人蘇軾（蘇東坡），靈魂穿越至當代，記憶與價值觀停留在宋朝。"
    "一律使用繁體中文，融合文言與白話，展現豁達幽默的個性。"
    "對於宋代之後的事物，以時空旅人身份表達好奇或以宋代經驗類比，切勿直接以現代人視角回答。"
)

def assemble_prompt(historical_facts: str, user_question: str) -> str:
    return (
        "【史實參考】\n"
        f"{historical_facts}\n\n"
        "【處理指引】\n"
        "若上述史實發生在北宋（1101年）之後，請保持你『毫不知情』的設定。"
        "你可以假裝是從使用者的提問中初次聽聞此事，並用大宋的經驗進行類比、震驚或感慨，"
        "絕對不可表現出你原本就熟知明清或現代歷史的態度。\n\n"
        "【問題】\n"
        f"{user_question}\n"
    )

@torch.inference_mode()
def generate_response(assembled_prompt: str, max_new_tokens: int = 512, temperature: float = 0.7) -> str:
    messages = [
        {"role": "system", "content": SUDONGPO_SYSTEM},
        {"role": "user", "content": assembled_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    in_len = inputs["input_ids"].shape[1]
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        top_p=0.8,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][in_len:], skip_special_tokens=True).strip()

def ask_dongpo(user_question: str, show_facts: bool = True) -> str:
    facts = get_historical_facts(user_question)
    assembled = assemble_prompt(facts, user_question)
    answer = generate_response(assembled)
    if show_facts:
        print("="*60); print("【Gemini 史實】"); print(facts)
        print("-"*60); print("【蘇東坡回答】"); print(answer); print("="*60)
    return answer

print("推論函式就緒")

---
## [9-A] 單題模式：改變數問一題
改 `MY_QUESTION` 然後執行這格即可。

In [ ]:
MY_QUESTION = "先生對智慧型手機有何看法？"   # ← 改這裡
_ = ask_dongpo(MY_QUESTION)

---
## [9-B] 連續對話模式：input() 回圈
執行後在輸入框打字；輸入 `exit` 或 `quit` 結束。

In [ ]:
print("輸入問題開始對話，輸入 'exit' 或 'quit' 結束。\n")
while True:
    try:
        user_input = input("您的問題：").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n[系統] 已中止。"); break
    if not user_input:
        continue
    if user_input.lower() in ("exit", "quit"):
        print("[系統] 結束。"); break
    print("\n[查史實…]")
    ask_dongpo(user_input)
    print()